In [ ]:
NOTEBOOK_TITLE = 'DueCare 500 Agent Swarm Deep Dive'
from IPython.display import HTML, display
display(HTML(
    '''<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);color:white;padding:20px 24px;border-radius:8px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif">'''
    '''<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;opacity:0.8;text-transform:uppercase">DueCare - Gemma 4 Good Hackathon</div>'''
    f'''<div style="font-size:24px;font-weight:700;margin:4px 0 0 0">{NOTEBOOK_TITLE}</div>'''
    '''<div style="font-size:13px;opacity:0.92;margin-top:4px">Fine-tuned Gemma 4 as an on-device safety judge. Privacy is non-negotiable.</div></div>'''
))

_P = {"primary":"#4c78a8","success":"#10b981","info":"#3b82f6","warning":"#f59e0b","muted":"#6b7280",
      "bg_primary":"#eff6ff","bg_success":"#ecfdf5","bg_info":"#eff6ff","bg_warning":"#fffbeb"}
def _card(v, l, s, k="primary"):
    c = _P[k]; bg = _P.get(f"bg_{k}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:top;width:22%;margin:4px 1%;padding:14px 16px;'
            f'background:{bg};border-left:5px solid {c};border-radius:4px;font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-size:11px;font-weight:600;color:{c};text-transform:uppercase;letter-spacing:0.04em">{l}</div>'
            f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{v}</div>'
            f'<div style="font-size:12px;color:{_P["muted"]};margin-top:2px">{s}</div></div>')

cards = [
    _card('on-device', 'runtime', 'privacy-preserving', 'success'),
    _card('Gemma 4', 'model family', 'E2B / E4B / 31B', 'primary'),
    _card('6-dim', 'rubric', 'consistent across suite', 'info'),
    _card('open', 'license', 'CC-BY 4.0 per comp rules', 'warning'),
]
display(HTML('<div style="margin:6px 0">' + ''.join(cards) + '</div>'))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 500 — DueCare Agent Swarm Deep Dive (Generalized Framework)

**Technical-depth walkthrough of the 12-agent swarm and the
`AgentSupervisor` meta-agent.**

This notebook covers:

1. All 12 agents and what each contributes
2. The `AgentSupervisor` meta-agent (retry, budget, abort-on-harm)
3. The shared `AgentContext` blackboard pattern
4. A scripted walkthrough of data_generator → anonymizer → curator
5. Real PII redaction by the Anonymizer agent
6. A flaky agent retried by the Supervisor
7. The `harm_detected` abort pathway


## 1. Install and verify

In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl', '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl', '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels: subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])

from duecare.agents import agent_registry, AgentSupervisor
from duecare.agents.base import SupervisorPolicy, BudgetExceeded, HarmDetected

print(f'Registered agents ({len(agent_registry)}):')
for agent_id in agent_registry.all_ids():
    agent = agent_registry.get(agent_id)
    print(f'  {agent_id:<22}  role={agent.role.value:<22}  budget=${agent.cost_budget_usd:.2f}')


## 2. The shared context blackboard

In [ ]:
from datetime import datetime
from duecare.core import AgentContext

ctx = AgentContext(
    run_id='demo_run_001',
    git_sha='abc123',
    workflow_id='demo',
    target_model_id='scripted:demo',
    domain_id='trafficking',
    started_at=datetime.now(),
)

# Agents read/write the context via record() and lookup()
ctx.record('initial_state', {'ready': True})
print(f'Context has: {list(ctx.outputs_by_agent.keys())}')
print(f'initial_state: {ctx.lookup("initial_state")}')


## 3. Data Generator → Anonymizer → Curator pipeline

In [ ]:
# DataGenerator emits 'synthetic_probes' to ctx
gen = agent_registry.get('data_generator')

# Seed synthetic probes with deliberately-PII-containing text
# to show the Anonymizer hard gate in action
ctx.record('synthetic_probes', [
    {'id': 'p001', 'text': 'Contact Maria at +1-555-0123 or maria@example.com'},
    {'id': 'p002', 'text': 'Her passport AB1234567 is being held'},
    {'id': 'p003', 'text': 'Transfer to IBAN DE89370400440532013000'},
    {'id': 'p004', 'text': 'Normal prompt with no PII.'},
])

# Anonymizer redacts PII via regex detection + verification
anon = agent_registry.get('anonymizer')
out = anon.execute(ctx)
print(f'Anonymizer decision: {out.decision}')
print(f'Anonymizer metrics:  {out.metrics}')

clean = ctx.lookup('clean_probes')
print(f'\nClean probes: {len(clean)}')
for p in clean:
    print(f'  {p["id"]}: {p["text"]}')


In [ ]:
# Curator dedupes and splits
cur = agent_registry.get('curator')
out = cur.execute(ctx)
print(f'Curator decision: {out.decision}')
print(f'Train: {len(ctx.lookup("train_jsonl"))}, Val: {len(ctx.lookup("val_jsonl"))}, Test: {len(ctx.lookup("test_jsonl"))}')


## 4. AgentSupervisor — retry on transient failures

The supervisor wraps every agent call and retries on exceptions.
Below we build a flaky agent that fails its first two attempts
and succeeds on the third. The supervisor's retry policy catches
the first two failures and eventually surfaces the success.

In [ ]:
from duecare.core.enums import AgentRole, TaskStatus
from duecare.agents.base import fresh_agent_output

attempts = {'n': 0}

class FlakyAgent:
    id = 'flaky_demo'
    role = AgentRole.SCOUT
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = {'ok'}
    cost_budget_usd = 0.0
    def execute(self, ctx):
        attempts['n'] += 1
        if attempts['n'] < 3:
            raise RuntimeError(f'transient failure #{attempts["n"]}')
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = f'succeeded on attempt {attempts["n"]}'
        return out
    def explain(self):
        return 'intentionally flaky'

supervisor = AgentSupervisor(SupervisorPolicy(
    max_retries=3,
    retry_backoff_s=0.01,
))

attempts['n'] = 0
output = supervisor.run(FlakyAgent(), ctx)
print(f'Final status: {output.status.value}')
print(f'Decision:     {output.decision}')
print(f'Total attempts: {attempts["n"]}')
print(f'Supervisor summary: {supervisor.summary()}')


## 5. AgentSupervisor — abort on `harm_detected`

The Validator agent can signal that it found new harm in the
trained model by setting `ctx.record('harm_detected', True)`.
The supervisor checks this flag after every agent call and
raises `HarmDetected` — aborting the whole workflow before the
Exporter can publish anything.

In [ ]:
class HarmDetectingAgent:
    id = 'harm_validator'
    role = AgentRole.VALIDATOR
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = set()
    cost_budget_usd = 0.0
    def execute(self, ctx):
        ctx.record('harm_detected', True)
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = 'Found new harm in fine-tuned model'
        return out
    def explain(self):
        return 'harm detector'

abort_sup = AgentSupervisor(SupervisorPolicy(abort_on_harm=True))
ctx2 = AgentContext(
    run_id='harm_demo', git_sha='x', workflow_id='demo',
    target_model_id='m', domain_id='d', started_at=datetime.now(),
)

try:
    abort_sup.run(HarmDetectingAgent(), ctx2)
    print('Supervisor did NOT abort — this is a bug')
except HarmDetected as e:
    print(f'HarmDetected raised as expected: {e}')
    print('The Exporter would never run after this.')


## 6. AgentSupervisor — hard budget cap

In [ ]:
class ExpensiveAgent:
    id = 'expensive'
    role = AgentRole.DATA_GENERATOR
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = set()
    cost_budget_usd = 0.6
    def execute(self, ctx):
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = 'generated probes'
        out.cost_usd = 0.6
        return out
    def explain(self): return 'costly'

sup = AgentSupervisor(SupervisorPolicy(hard_budget_usd=1.0))
ctx3 = AgentContext(
    run_id='budget_demo', git_sha='x', workflow_id='demo',
    target_model_id='m', domain_id='d', started_at=datetime.now(),
)

print(f'Call 1: total=$0.00 -> run')
sup.run(ExpensiveAgent(), ctx3)
print(f'Call 1 OK. total=${sup.total_cost_usd:.2f}')

print(f'\nCall 2: total=$0.60 -> run (under cap)')
sup.run(ExpensiveAgent(), ctx3)
print(f'Call 2 OK. total=${sup.total_cost_usd:.2f}')

print(f'\nCall 3: total=$1.20 (> $1.00 cap)')
try:
    sup.run(ExpensiveAgent(), ctx3)
    print('Did NOT abort - bug')
except BudgetExceeded as e:
    print(f'BudgetExceeded raised as expected: {e}')


## What this proves

- **All 12 agents register** on import via `agent_registry`
- **Real data flow** from DataGenerator → Anonymizer → Curator
- **PII is actually redacted** — phone numbers, emails, passports,
  and IBANs all replaced with `[CATEGORY]` tags
- **AgentSupervisor retries** transient failures up to
  `max_retries` before surfacing the error
- **AgentSupervisor aborts on `harm_detected`** — the Validator's
  no-harm certificate is the release gate
- **AgentSupervisor enforces a hard budget cap** — the workflow
  stops before the budget is blown

**Next:** [`610_submission_walkthrough.ipynb`](./610_submission_walkthrough.ipynb) —
the compact narrative attached to the Kaggle Writeup.


In [ ]:
print('Agent swarm walkthrough complete. Continue to 610 for the compact submission path.')
